<a href="https://colab.research.google.com/github/NguyenNguyen1504/GROUP-5-BPFE/blob/main/GROUP_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **I. Load and clean datasets**



### **Setup path and read input**



In [57]:
from google.colab import drive
import os
import pandas as pd

drive.mount('/content/drive')

base_dataset_path = '/content/drive/MyDrive/EPL_data/results.csv'

if os.path.exists(base_dataset_path):
    df_results = pd.read_csv(base_dataset_path, encoding="latin1") # Base dataset
    print("Read data successfully.")
    display(df_results.head())
else:
    print("Error: file not found.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Read data successfully.


,Season,DateTime,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,HTR,...,HST,AST,HC,AC,HF,AF,HY,AY,HR,AR
0,1993-94,1993-08-14T00:00:00Z,Arsenal,Coventry,0,3,A,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1993-94,1993-08-14T00:00:00Z,Aston Villa,QPR,4,1,H,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1993-94,1993-08-14T00:00:00Z,Chelsea,Blackburn,1,2,A,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1993-94,1993-08-14T00:00:00Z,Liverpool,Sheffield Weds,2,0,H,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1993-94,1993-08-14T00:00:00Z,Man City,Leeds,1,1,D,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### **Clean data**

In [58]:
columns_to_keep = ['Season', 'DateTime', 'HomeTeam', 'AwayTeam', 'FTHG', 'FTAG', 'FTR']
# 93-94 and 94-95 seasons had 22 teams
rows_to_drop = ['1993-94', '1994-95']
df_results = df_results[columns_to_keep].copy()
df_results = df_results[~df_results['Season'].isin(rows_to_drop)].copy()

### **Create dictionaries saving data of clubs' home stadiums capacities**

**Source: Wikipedia, NBC Sports** (https://www.nbcsports.com/soccer/news/list-of-premier-league-stadiums-every-clubs-current-and-former-ground-from-the-pl-era)

In [59]:
# Teams with changed stadiums
changes = {
    'Arsenal':       [('2006-07', 60704), ('1993-94', 38419)],
    'Man City':      [('2003-04', 53400), ('1993-94', 35150)],
    'Southampton':   [('2001-02', 32384), ('1993-94', 15200)],
    'Leicester':     [('2002-03', 32261), ('1993-94', 22000)],
    'Middlesbrough': [('1995-96', 34742), ('1993-94', 26667)],
    'West Ham':      [('2016-17', 60000), ('1993-94', 35016)],
    'Tottenham':     [('2019-20', 62850), ('1993-94', 36284)],
    'Bolton':        [('1997-98', 28723), ('1993-94', 22000)],
    'Sunderland':    [('1997-98', 49000), ('1993-94', 22000)]
}

# Teams with unchanged stadiums
fixed = {
    'Man United':       74197,
    'Liverpool':        54074,
    'Newcastle':        52258,
    'Aston Villa':      42785,
    'Chelsea':          40173,
    'Everton':          39414,
    'Sheffield Weds':   39812,
    'Leeds':            37890,
    'Blackburn':        31367,
    'Wolves':           32050,
    'Sheffield United': 32050,
    'Brighton':         31876,
    'Ipswich':          29813,
    'Birmingham':       29409,
    'Stoke':            27902,
    'Norwich':          27359,
    'Charlton':         27111,
    'West Brom':        26850,
    'Wimbledon':        26309,
    'Fulham':           25700,
    'Crystal Palace':   25486,
    'Hull':             25400,
    'Wigan':            25138,
    'Bradford':         25136,
    'Huddersfield':     24500,
    'Reading':          24161,
    'Barnsley':         23009,
    'Watford':          22200,
    'Burnley':          21944,
    'Swansea':          21088,
    'Portsmouth':       20700,
    'QPR':              18360,
    'Brentford':        17250,
    'Blackpool':        16220,
    'Swindon':          15728,
    'Oldham':           13512,
    'Bournemouth':      11307,
    'Derby':            33597,
    'Cardiff':          33280,
    'Coventry':         32609,
    "Nott'm Forest":    30404
}

### **Create a column for home stadium capacity in the big dataset**

In [60]:
def get_capacity(team, season):

    if team in changes:
        for from_season, cap in changes[team]:
            if season >= from_season:
                return cap

    if team in fixed:
        return fixed[team]

    print(f"WARNING: No capacity data for team '{team}'")
    return None

df_results['HomeTeam_StadiumCapacity'] = df_results.apply(lambda row: get_capacity(row['HomeTeam'],row['Season']), axis=1)
df_results = df_results.reset_index(drop=True)

display(df_results.head())

,Season,DateTime,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HomeTeam_StadiumCapacity
0,1995-96,1995-08-19T00:00:00Z,Aston Villa,Man United,3,1,H,42785
1,1995-96,1995-08-19T00:00:00Z,Blackburn,QPR,1,0,H,31367
2,1995-96,1995-08-19T00:00:00Z,Chelsea,Everton,0,0,D,40173
3,1995-96,1995-08-19T00:00:00Z,Liverpool,Sheffield Weds,1,0,H,54074
4,1995-96,1995-08-19T00:00:00Z,Man City,Tottenham,1,1,D,35150


In [61]:
df_results['CapacityGroup'] = pd.cut(
    df_results['HomeTeam_StadiumCapacity'],
    bins = 3,
    labels = ['Small', 'Medium', 'Large']
)

### **Assign the matches to the COVID period**

In [67]:
def covid_period(season):
  if season < '2020-21':
    return 'pre-COVID'
  elif season == '2020-21':
    return 'COVID'
  else:
    return 'post-COVID'

df_results['COVID_period'] = df_results['Season'].apply(lambda season: covid_period(season))
display(df_results.head(10))

,Season,DateTime,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HomeTeam_StadiumCapacity,CapacityGroup,COVID_period
0,1995-96,1995-08-19T00:00:00Z,Aston Villa,Man United,3,1,H,42785,Medium,pre-COVID
1,1995-96,1995-08-19T00:00:00Z,Blackburn,QPR,1,0,H,31367,Small,pre-COVID
2,1995-96,1995-08-19T00:00:00Z,Chelsea,Everton,0,0,D,40173,Medium,pre-COVID
3,1995-96,1995-08-19T00:00:00Z,Liverpool,Sheffield Weds,1,0,H,54074,Large,pre-COVID
4,1995-96,1995-08-19T00:00:00Z,Man City,Tottenham,1,1,D,35150,Medium,pre-COVID
5,1995-96,1995-08-19T00:00:00Z,Newcastle,Coventry,3,0,H,52258,Medium,pre-COVID
6,1995-96,1995-08-19T00:00:00Z,Southampton,Nott'm Forest,3,4,A,15200,Small,pre-COVID
7,1995-96,1995-08-19T00:00:00Z,West Ham,Leeds,1,2,A,35016,Medium,pre-COVID
8,1995-96,1995-08-19T00:00:00Z,Wimbledon,Bolton,3,2,H,26309,Small,pre-COVID
9,1995-96,1995-08-20T00:00:00Z,Arsenal,Middlesbrough,1,1,D,38419,Medium,pre-COVID


# **Calculate statistics**

### **Calculate Home win rate for each club across seasons**

In [63]:
df_hwin_clubs = df_results.groupby(['Season','HomeTeam']).agg(
    HomeWinRate = ('FTR', lambda col: (col == 'H').mean()),
    StadiumCapacity = ('HomeTeam_StadiumCapacity', 'first')
).reset_index()
display(df_hwin_clubs.head())

,Season,HomeTeam,HomeWinRate,StadiumCapacity
0,1995-96,Arsenal,0.526316,38419
1,1995-96,Aston Villa,0.578947,42785
2,1995-96,Blackburn,0.736842,31367
3,1995-96,Bolton,0.263158,22000
4,1995-96,Chelsea,0.368421,40173


### **Calculate Home win rate - Away win rate by capacity group**

In [64]:
df_hwin_awin = df_results.groupby(['CapacityGroup']).agg(
    HomeWinRate = ('FTR', lambda col: (col == 'H').mean()),
    AwayWinRate = ('FTR', lambda col: (col == 'A').mean())
).reset_index()
display(df_hwin_awin.head())

/tmp/ipykernel_18814/3865357821.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df_hwin_awin = df_results.groupby(['CapacityGroup']).agg(


,CapacityGroup,HomeWinRate,AwayWinRate
0,Small,0.365619,0.364858
1,Medium,0.468211,0.270436
2,Large,0.644758,0.152091


### **Calculate Home win rate by capacity group across seasons**

In [65]:
df_hwin_seasons = df_results.groupby(['Season','CapacityGroup']).agg(
    HomeWinRate = ('FTR', lambda col: (col == 'H').mean())
).reset_index()
display(df_hwin_seasons.head())

/tmp/ipykernel_18814/2618090140.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df_hwin_seasons = df_results.groupby(['Season','CapacityGroup']).agg(


,Season,CapacityGroup,HomeWinRate
0,1995-96,Small,0.421053
1,1995-96,Medium,0.478070
2,1995-96,Large,0.763158
3,1996-97,Small,0.350877
4,1996-97,Medium,0.438596


### **Calculate Home win rate by capacity group across COVID periods**

In [66]:
df_hwin_covid = df_results.groupby(['COVID_period','CapacityGroup']).agg(
    HomeWinRate = ('FTR', lambda col: (col == 'H').mean())
).reset_index()
display(df_hwin_covid.head(9))

/tmp/ipykernel_18814/4214052617.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df_hwin_covid = df_results.groupby(['COVID_period','CapacityGroup']).agg(


,COVID_period,CapacityGroup,HomeWinRate
0,COVID,Small,0.263158
1,COVID,Medium,0.385965
2,COVID,Large,0.526316
3,post-COVID,Small,0.296000
4,post-COVID,Medium,0.362637
5,post-COVID,Large,0.645161
6,pre-COVID,Small,0.372239
7,pre-COVID,Medium,0.472732
8,pre-COVID,Large,0.652999
